In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("env", "dev", "Ambiente (dev | prod)")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog_base", "catalog_pr_db")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsdvbtdev01")


env = dbutils.widgets.get("env").strip().lower()
container = dbutils.widgets.get("container")
catalogo = f"{dbutils.widgets.get('catalog_base')}_{env}"
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
 
assert env in ("dev", "prod"), f"env inválido: {env}"
 
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/results.csv"
print(f"env={env} | catalogo={catalogo} | ruta={ruta}")


In [0]:
df_results= spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
df_results.display()

In [0]:
results_schema = StructType([
    StructField("resultId", IntegerType(), False),
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("number", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("position", StringType(), True),
    StructField("positionText", StringType(), True),
    StructField("positionOrder", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("laps", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("milliseconds", StringType(), True),
    StructField("fastestLap", StringType(), True),
    StructField("rank", StringType(), True),
    StructField("fastestLapTime", StringType(), True),
    StructField("fastestLapSpeed", StringType(), True),
    StructField("statusId", IntegerType(), True)
])

In [0]:
results_schema

In [0]:
results_df= spark.read \
            .option("header", True) \
            .schema(results_schema) \
            .csv(ruta)

In [0]:
results_with_timestamp_df = results_df.withColumn("ingestion_date", current_timestamp())

In [0]:
results_selected_df = results_with_timestamp_df.select(col("resultId").alias('result_id'),
    col("raceId").alias('race_id'),
    col("driverId").alias('driver_id'),
    col("constructorId").alias('constructor_id'),
    col("number").alias('number_results'),
    col("grid"),
    col("position"),
    col("positionText").alias('position_text'),
    col("positionOrder").alias('position_order'),
    col("points"),
    col("laps"),
    col("time"),
    col("milliseconds"),
    col("fastestLap").alias('fastest_lap'),
    col("rank"),
    col("fastestLapTime").alias('fastest_lap_time'),
    col("fastestLapSpeed").alias('fastest_lap_speed'),
    col("statusId").alias('status_id'),
    col("ingestion_date").alias('ingestion_date'))

In [0]:
display(results_selected_df)

In [0]:
print(f"catalogo = {catalogo}")
print(f"esquema = {esquema}")

In [0]:
spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"USE SCHEMA `{esquema}`")

In [0]:
print(f"current_catalog = {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"current_schema = {spark.sql('SELECT current_schema()').collect()[0][0]}")


In [0]:
results_selected_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.results")

In [0]:
%sql
select * FROM catalog_pr_db_dev.bronze.results